# VN30F1M Momentum Transformer (Fast Colab Run)

Clone from GitHub, run main model config, and auto-download outputs.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/votranhuonggiang/Trading-with-the-Momentum-Transformer.git'
WORKDIR = '/content/Trading-with-the-Momentum-Transformer'

import os, shutil
from pathlib import Path

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.system(f'git clone {GITHUB_REPO_URL} {WORKDIR}')

csv_path = Path(WORKDIR) / 'data' / 'raw' / 'vn30f1m.csv'
assert csv_path.exists(), f'Missing required file: {csv_path}'
print('Found input:', csv_path)

In [ ]:
%cd /content/Trading-with-the-Momentum-Transformer
!python -m pip install -U pip
!pip install -q pandas numpy pyyaml pyarrow scikit-learn matplotlib torch

In [ ]:
from pathlib import Path
import yaml

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))

# Core paths + trading cost
cfg['paths']['project_root'] = '/content/Trading-with-the-Momentum-Transformer'
cfg['paths']['raw_data_csv'] = 'data/raw/vn30f1m.csv'
cfg['trading']['base_round_trip_cost_points'] = 0.20
cfg['trading']['cost_scenarios_points'] = [0.20]

# Main model only + fast runtime profile
cfg['models']['main_models'] = ['decoder_tft']
cfg['walk_forward']['max_windows'] = 1
cfg['training']['max_epochs_per_window'] = 1
cfg['training']['max_train_rows_per_window'] = 5000
cfg['training']['max_valid_rows_per_window'] = 2000

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Config updated for fast main-model run')

In [ ]:
!python src/data_preprocessing.py --config configs/default.yaml
!python src/feature_engineering.py --config configs/default.yaml
!python src/walk_forward.py --config configs/default.yaml
!python src/evaluate.py --config configs/default.yaml

In [ ]:
!ls -lah outputs/metrics
!ls -lah outputs/tables

In [ ]:
# Auto-download outputs to your laptop
import shutil
from google.colab import files

zip_base = '/content/vn30f1m_outputs'
zip_file = shutil.make_archive(zip_base, 'zip', root_dir='outputs')
print('Created:', zip_file)
files.download(zip_file)